In [1]:
import sys
sys.path.insert(0, "../driving-models")

In [2]:
import os
import logging
import math
import inspect
import json
import random
import numpy as np
from dataclasses import dataclass, field
from typing import Optional, List, Dict

import torch
import gymnasium as gym

from torchdrivesim.goals import WaypointGoal
from torchdrivesim.kinematic import KinematicBicycle
from torchdrivesim.rendering import renderer_from_config
from torchdrivesim.rendering.base import RendererConfig
from torchdrivesim.utils import Resolution
from torchdrivesim.lanelet2 import find_lanelet_directions
from torchdrivesim.map import find_map_config, traffic_controls_from_map_config
from torchdrivesim.traffic_lights import current_light_state_tensor_from_controller
from torchdrivesim.simulator import TorchDriveConfig, SimulatorInterface, \
    BirdviewRecordingWrapper, Simulator, CollisionMetric
from torchdrivesim.behavior.replay import ReplayController

No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


In [3]:
from driving_models.data.iai.sqlite_dataset import IAISQLiteDataset, IAISQLiteDatasetConfig

from driving_models.data.common import ObjectType, DatasetSplit, ArgoverseTrackCategory

import torch
import numpy as np
import matplotlib.pyplot as plt
from driving_models.trainers.utils import cast_to_device

In [4]:
logger = logging.getLogger(__name__)

In [6]:
@dataclass
class EnvConfig:
    ego_only: bool = False
    max_environment_steps: int = 200
    frame_stack: int = 3
    waypoint_bonus: float = 100.
    heading_penalty: float = 25.
    distance_bonus: float = 1.
    distance_cutoff: float = 0.5
    use_background_traffic: bool = True
    terminated_at_infraction: bool = True
    seed: Optional[int] = None
    simulator: TorchDriveConfig = field(default_factory=lambda:TorchDriveConfig(renderer=RendererConfig(left_handed_coordinates=False,
                                                                           highlight_ego_vehicle=True),
                                                   collision_metric=CollisionMetric.nograd,
                                                   left_handed_coordinates=False))
    render_mode: Optional[str] = "rgb_array"
    video_filename: Optional[str] = "rendered_video.mp4"
    video_res: Optional[int] = 1024
    video_fov: Optional[float] = 500
    device: Optional[str] = None

In [7]:
def save_video(imgs, filename, batch_index=0, fps=10, web_browser_friendly=False):
    import cv2
    img_stack = [cv2.cvtColor(
        img[batch_index].cpu().numpy().astype(
            np.uint8).transpose(1, 2, 0), cv2.COLOR_RGB2BGR
    )
        for img in imgs]

    w = img_stack[0].shape[0]
    h = img_stack[0].shape[1]
    output_format = cv2.VideoWriter_fourcc(*'mp4v')

    vid_out = cv2.VideoWriter(filename=filename,
                              fourcc=output_format,
                              fps=fps,
                              frameSize=(w, h))

    for frame in img_stack:
        vid_out.write(frame)

    vid_out.release()

    if web_browser_friendly:
        import uuid
        temp_filename = os.path.join(os.path.dirname(
            filename), str(uuid.uuid4()) + '.mp4')
        os.rename(filename, temp_filename)
        os.system(
            f"ffmpeg -y -i {temp_filename} -hide_banner -loglevel error -vcodec libx264 -f mp4 {filename}")
        os.remove(temp_filename)


def set_seeds(seed, logger=None):
    if seed is None:
        seed = np.random.randint(low=0, high=2**32 - 1)
    if logger is not None:
        logger.info(f"seed: {seed}")
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    return seed

In [8]:
BASE_PATH = "../../data/train_data/"

In [9]:
dataset_cfg = IAISQLiteDatasetConfig(
        url=f'file://{BASE_PATH}',
        include_aerial_image=False,
        custom_segment_collection_name='argoverse2',
        segment_length=110,
        segment_spacing=0,
#         ego_types=None,
        return_lanelet_map=False,
        recenter=False,
        include_lane_boundary_segments=False,
        use_fixed_agent_type_names=True
    )
sql_dataset = IAISQLiteDataset(
        cfg=dataset_cfg, split=DatasetSplit.train
    )

INFO:driving_models.data.iai.sqlite_dataset:Using stored SegmentCollection 2 named argoverse2
INFO:driving_models.data.iai.sqlite_dataset:Loaded 176740 for split train


In [16]:
def build_simulator(cfg: EnvConfig, data, device):
    with torch.no_grad():
        driving_surface_mesh = data['driving_surface_mesh']
        birdview_mesh_generator = BirdviewRGBMeshGenerator()        
        
        # size: 1 (batch_size) x 1 (ego_agent) x 1 (timestep=0) x 4
        ego_idx = data['ego_idx']
        ego_agent_state = data['state'][ego_idx, 0, :].unsqueeze(0)
        ego_initial_present_mask = data['present_mask'][ego_idx, 0].unsqueeze(0)
        ego_agent_size = data['lenwid'][ego_idx, :].unsqueeze(0)
        ego_agent_type = data['agent_types'][ego_idx].unsqueeze(0)
        ego_lr = data['lr'][ego_idx].unsqueeze(0)
        
        
        # size: 1 (batch_size) x (A - 1) (npc_agents) x T (timesteps) x 4
        npc_states = torch.cat((data['state'][:ego_idx, ...], data['state'][ego_idx + 1:, ...]), dim=0).unsqueeze(0)
        npc_present_masks = torch.cat((data['present_mask'][:ego_idx, ...], data['present_mask'][ego_idx + 1:, ...]), dim=0).unsqueeze(0)
        npc_size = torch.cat((data['lenwid'][:ego_idx, ...], data['lenwid'][ego_idx + 1:, ...]), dim=0).unsqueeze(0)
        npc_types = torch.cat((data['agent_types'][:ego_idx, ...], data['agent_types'][ego_idx + 1:, ...]), dim=0).unsqueeze(0)

        kinematic_model = KinematicBicycle()
        kinematic_model.set_params(lr=ego_lr)
        kinematic_model.set_state(ego_agent_state)
        
        
        npc_controller = ReplayController(npc_size=npc_size, 
                                                                   npc_states=npc_states, 
                                                                   npc_present_masks=npc_present_masks, 
                                                                   npc_types=npc_types, 
                                                                   agent_type_names=data['agent_type_names'])
        
        renderer = renderer_from_config(cfg.simulator.renderer)

        simulator = Simulator(
            cfg=cfg.simulator, 
            road_mesh=driving_surface_mesh,
            initial_present_mask=ego_initial_present_mask,
            kinematic_model=kinematic_model, 
            agent_size=ego_agent_size,
            agent_types=ego_agent_type,
            agent_type_names=data['agent_type_names'],
            renderer=renderer,
            npc_controller=npc_controller
        )

        if cfg.render_mode == "video":
            simulator = BirdviewRecordingWrapper(
                simulator, res=Resolution(cfg.video_res, cfg.video_res), fov=cfg.video_fov, to_cpu=True)
        simulator.to(device)

        return simulator

In [19]:
class GymEnv(gym.Env):

    metadata = {
        "render_modes": ["video", "rgb_array"],
        "render_fps": 10
    }

    def __init__(self, cfg: EnvConfig, simulator: SimulatorInterface):
        if cfg.render_mode is not None and cfg.render_mode not in self.metadata["render_modes"]:
            raise NotImplementedError
        self.render_mode = cfg.render_mode

        acceleration_range = (-1.0, 1.0)
        steering_range = (-0.3, 0.3)
        action_range = np.ndarray(shape=(2, 2), dtype=np.float32)
        action_range[:, 0] = acceleration_range
        action_range[:, 1] = steering_range
        self.max_environment_steps = cfg.max_environment_steps
        self.environment_steps = 0
        self.action_space = gym.spaces.Box(
            low=action_range[0],
            high=action_range[1],
            dtype=np.float32
        )
        self.observation_space = gym.spaces.Box(low=0, high=255, shape=(3, 64, 64), dtype=np.uint8)

        self.reward_range = (- float('inf'), float('inf'))
        self.collision_threshold = 0.0
        self.offroad_threshold = 0.0

        self.config = cfg
        self.simulator = simulator
        self.current_action = None

        self.last_birdview = None
        
    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        self.simulator = self.start_sim.copy()
        self.environment_steps = 0
        self.last_birdview = None
        return self.get_obs(), {}

    def step(self, action: np.array):
        self.environment_steps += 1
        self.simulator.step(action)
        self.last_action = self.current_action if self.current_action is not None else action
        self.current_action = action
        return self.get_obs(), self.get_reward(), self.is_terminated(), self.is_truncated(), self.get_info()

    def get_obs(self):
        birdview = self.simulator.render_egocentric().cpu().numpy().astype(np.uint8)
        return birdview

    def get_reward(self):
        x = self.simulator.get_state()[..., 0]
        r = np.zeros(x.shape)
        return r

    def is_done(self):
         return self.is_truncated() or self.is_terminated()

    def is_truncated(self):
        return self.environment_steps >= self.max_environment_steps

    def is_terminated(self):
        return False

    def get_info(self):
        self.info = dict(
            offroad=self.simulator.compute_offroad(),
            collision=self.simulator.compute_collision(),
            traffic_light_violation=self.simulator.compute_traffic_lights_violations(),
            is_success=(self.environment_steps >= self.max_environment_steps),
        )
        return self.info

    def seed(self, seed=None):
        pass

    def render(self):
        if self.render_mode == 'rgb_array':
            birdview = self.simulator.render_egocentric().cpu().numpy()
            return np.transpose(birdview.squeeze(), axes=(1, 2, 0))
        else:
            raise NotImplementedError

    def close(self):
        if isinstance(self.simulator, BirdviewRecordingWrapper):
            bvs = self.simulator.get_birdviews()
            if len(bvs) > 1:
                save_video(bvs, self.config.video_filename)

In [20]:
class ArgoEnv(GymEnv):
    def __init__(self, cfg: EnvConfig, dataset: IAISQLiteDataset):
        self.config = cfg
        self.dataset = dataset
        self.dataset_size = len(dataset)
        
        if cfg.device is None:
            self.torch_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        else:
            self.torch_device = torch.device(cfg.device)

        set_seeds(self.config.seed, logger)
        
        super().__init__(cfg=cfg, simulator=None)

    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        self.current_data_idx = np.random.randint(self.dataset_size)
        self.data = self.dataset[self.current_data_idx]

        self.last_x = None
        self.last_y = None
        self.last_psi = None

        self.last_obs = None
        self.last_reward = None
        self.last_info = None

        self.environment_steps = 0
        
        self.simulator = build_simulator(self.config,
                                                                self.data,
                                                                device=self.torch_device)

        return self.get_obs(), {}

        
    def step(self, action: np.array):
        state = self.simulator.get_state()
        self.last_x = state[..., 0]
        self.last_y = state[..., 1]
        self.last_psi = state[..., 2]
        self.last_speed = state[..., 3]

        obs, reward, terminated, truncated, info = super().step(action)

        self.last_obs = obs
        self.last_reward = reward
        self.last_info = info
        return obs, reward, terminated, truncated, info
 

    def get_reward(self):
        x = self.simulator.get_state()[..., 0]
        y = self.simulator.get_state()[..., 1]
        psi = self.simulator.get_state()[..., 2]

        d = math.dist((x, y), (self.last_x, self.last_y)) if (self.last_x is not None) and (self.last_y is not None) else 0
        distance_reward = self.config.distance_bonus if d > self.config.distance_cutoff else 0
        psi_reward = (1 - math.cos(psi - self.last_psi)) * (- self.config.heading_penalty) if (self.last_psi is not None) else 0
        r = torch.zeros_like(x)
        r += distance_reward + psi_reward
        return r.item()

    def is_terminated(self):       
        if self.config.terminated_at_infraction:
            return ((self.simulator.compute_offroad() > 0) or (self.simulator.compute_collision() > 0)).item()
        else:
            return False
    
    def get_info(self):
        x = self.simulator.get_state()[..., 0]
        y = self.simulator.get_state()[..., 1]
        psi = self.simulator.get_state()[..., 2]
        speed = self.simulator.get_state()[..., 3]
        d = math.dist((x, y), (self.last_x, self.last_y)) if (self.last_x is not None) and (self.last_y is not None) else 0
        # reached_waypoint_num = self.reached_waypoint_num
        self.info = dict(
            offroad=self.simulator.compute_offroad(),
            collision=self.simulator.compute_collision(),
            is_success=(self.environment_steps >= self.max_environment_steps),
            psi_smoothness=((self.last_psi - psi) / 0.1).norm(p=2).item(),
            psi_reward=(1 - math.cos(psi - self.last_psi)) * (- self.config.heading_penalty),
            dist_reward=self.config.distance_bonus if d > self.config.distance_cutoff else 0,
            speed_smoothness=((self.last_speed - speed) / 0.1).norm(p=2).item()
        )
        return self.info
    
            # reached_waypoint_num=reached_waypoint_num,
            # traffic_light_violation=self.simulator.compute_traffic_lights_violations(),

In [21]:
env_config = EnvConfig()
env_config.render_mode="video"

In [22]:
env = ArgoEnv(cfg=env_config, dataset=sql_dataset)

INFO:__main__:seed: 203068099


In [23]:
env.reset()
while True:
    actions = torch.tensor([[[1, 0]]], dtype=torch.float32, device='cpu')  # accelerate hard without steering
    obs, reward, terminated, truncated, info = env.step(actions)  
    print(reward)
    print(info)
    if terminated or truncated:
        break
env.close()

1.0
{'offroad': tensor([[0.]]), 'collision': tensor([[0.]], dtype=torch.float64), 'is_success': False, 'psi_smoothness': 0.0, 'psi_reward': -0.0, 'dist_reward': 1.0, 'speed_smoothness': 5.0}
1.0
{'offroad': tensor([[0.]]), 'collision': tensor([[0.]], dtype=torch.float64), 'is_success': False, 'psi_smoothness': 0.0, 'psi_reward': -0.0, 'dist_reward': 1.0, 'speed_smoothness': 5.0}
1.0
{'offroad': tensor([[0.]]), 'collision': tensor([[0.]], dtype=torch.float64), 'is_success': False, 'psi_smoothness': 0.0, 'psi_reward': -0.0, 'dist_reward': 1.0, 'speed_smoothness': 5.0}
1.0
{'offroad': tensor([[0.]]), 'collision': tensor([[0.]], dtype=torch.float64), 'is_success': False, 'psi_smoothness': 0.0, 'psi_reward': -0.0, 'dist_reward': 1.0, 'speed_smoothness': 5.0}
1.0
{'offroad': tensor([[0.]]), 'collision': tensor([[0.]], dtype=torch.float64), 'is_success': False, 'psi_smoothness': 0.0, 'psi_reward': -0.0, 'dist_reward': 1.0, 'speed_smoothness': 5.0}
1.0
{'offroad': tensor([[0.]]), 'collision': 